In [ ]:
# -*- coding: utf-8 -*-
"""
DRM Rolling-Average Pipeline
=============================
Folder-based, catalyst-agnostic pipeline. Drop any number of
"<ID>_bypass.dat" / "<ID>_reaction.dat" / "<ID>_parameters.dat" triplets
into INPUT_FOLDER and this script will discover, process, and compare
all of them - no hardcoded catalyst names or file paths.

Pipeline stages (in the order requested):
  1. READ         - locate + parse each catalyst's bypass/reaction/parameters files
  2. MERGE         - interpolate parameters onto the reaction time axis and
                     merge_asof (nearest) onto the reaction file WITHOUT
                     touching any of the reaction file's own values.
                     NOTE: parameter columns (TIR-1, TIR-2, PIR-1, MFC-*, ...)
                     are NEVER rolling-averaged/smoothed anywhere in this
                     script - only the Sig:_MCD_* mass-spec species are.
  3. ROLLING AVG   - smooth the pulsing Sig:_MCD_* species (CH4, CO2, Ar, H2,
                     CO, H2O) using a window sized to a whole number of
                     pulse cycles (auto-detected from the CH4 pulse period,
                     per file)
  4. QC PLOTS      - raw vs rolling-average, generated (and shown) BEFORE
                     any KPI is calculated, per catalyst:
                       - bypass: full run (it's short)
                       - reaction: start / middle / end zoom windows
  5. KPIs          - conversions / yields / selectivities / carbon balance,
                     computed from the ROLLING-AVERAGED signals (not raw)
  6. DASHBOARD     - single compiled HTML with two sections:
                       - TIME-DEPENDENT PLOTS   (x = time, dual y-axis with
                         Temperature on the right, like the old "stability"
                         plot - now covers every KPI, not just CO yield)
                       - TEMPERATURE-DEPENDENT PLOTS (x = Temperature,
                         single y-axis - the original 5 comparison rows)
                     Both sections also embed the QC plots (raw vs rolling
                     avg) at the top so the smoothing quality is visible
                     right next to the KPI results it feeds.

Outputs (all under OUTPUT_FOLDER):
    <ID>_integrated_results.xlsx   (per catalyst, 3 sheets)
    <ID>_averaging_qc_reaction.png (per catalyst)
    <ID>_averaging_qc_bypass.png   (per catalyst)
    Master_DRM_LookerStudio_Data.csv
    DRM_Interactive_Dashboard.html
"""

import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =============================================================================
# 0. CONFIGURATION
# =============================================================================
INPUT_FOLDER = r"C:\Users\kwind\OneDrive\Documents\Data analytics Bootcamp\Project 2 pulsing\All catalysts"     # <- point this at your data folder
OUTPUT_FOLDER = r"C:\Users\kwind\OneDrive\Documents\Data analytics Bootcamp\Project 2 pulsing\All catalysts"    # <- where results get written

# Species to smooth / analyse. Ar is the internal-standard / carrier gas -
# included per request. Extend this dict if new species need tracking; the
# rest of the pipeline (rolling avg + QC plots) needs no other changes.
SPECIES = {
    "Sig:_MCD_CH4": "CH4",
    "Sig:_MCD_CO2": "CO2",
    "Sig:_MCD_Ar":  "Ar",
    "Sig:_MCD_H2":  "H2",
    "Sig:_MCD_CO":  "CO",
    "Sig:_MCD_H2O": "H2O",
}
COLORS = {
    "CH4": "tab:green", "CO2": "tab:red", "Ar": "tab:gray",
    "H2": "tab:blue", "CO": "tab:orange", "H2O": "tab:purple",
}

# Rolling-average window = N_CYCLES full pulse periods (auto-detected from
# the Ar sweep-gas signal - see PULSE_REFERENCE_SPECIES and detect_pulse_period()).
N_CYCLES = 2

# Minimum allowed spacing (seconds) between two detected pulse peaks - just
# a safety filter so one noisy pulse plateau isn't counted as multiple
# peaks. It only needs to be smaller than the real pulse period, not equal
# to it. RULE OF THUMB if you change the pulsing timing: set this to
# roughly 30-50% of your new expected pulse period.
#   e.g. shorter ~15s pulses  -> MIN_PEAK_GAP_SEC ~= 5-7
#        longer  ~60s pulses  -> MIN_PEAK_GAP_SEC ~= 15-20 (10 still works)
MIN_PEAK_GAP_SEC = 10

# Species used as the reference channel for pulse-period detection. Ar is
# the sweep/inert gas, so it pulses (inverted) regardless of WHICH reactant
# is being cycled (CH4, CO2, or swapped later) - this makes period detection
# independent of any future change to reactant pulsing choice. Verified: an
# asymmetric duty cycle between reactant and Ar does NOT change the measured
# period (both channels share the same underlying valve-switching clock) -
# the only real risk is if a phase's "on" time drops below ~1-2 sample
# intervals, which is far shorter than any realistic pulse timing.
PULSE_REFERENCE_SPECIES = "Sig:_MCD_Ar"

# Fraction of the total reaction run shown in each of the start/mid/end
# QC zoom windows.
ZOOM_WIDTH_FRAC = 0.05

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# =============================================================================
# 1. FILE DISCOVERY
# =============================================================================
def discover_catalysts(folder):
    """Find every '<ID>_reaction.dat'-style file in `folder` and return the
    catalyst IDs that have a matching bypass + parameters file too. Handles
    both '<ID>_reaction.dat' and '<ID> reaction.dat' naming."""
    candidates = glob.glob(os.path.join(folder, "*reaction.dat"))
    ids = []
    for f in candidates:
        base = os.path.basename(f)
        core = re.sub(r"[_ ]?reaction\.dat$", "", base, flags=re.IGNORECASE)
        ids.append(core)
    ids = sorted(set(ids))

    complete = []
    for cat_id in ids:
        for sep in ("_", " "):
            b = os.path.join(folder, f"{cat_id}{sep}bypass.dat")
            r = os.path.join(folder, f"{cat_id}{sep}reaction.dat")
            p = os.path.join(folder, f"{cat_id}{sep}parameters.dat")
            if os.path.exists(b) and os.path.exists(r) and os.path.exists(p):
                complete.append({"id": cat_id, "bypass": b, "reaction": r, "parameters": p})
                break
    return complete


# =============================================================================
# 2. PARSING UTILITIES
# =============================================================================
def load_mass_spec_file(file_path):
    """Parse a Pfeiffer PrismaPro export. Finds the header row dynamically
    (rather than assuming a fixed skiprows count) so it survives export
    format changes."""
    with open(file_path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.readlines()
    header_idx = next((i for i, ln in enumerate(lines) if ln.startswith("Time Relative (sec)")), None)
    if header_idx is None:
        raise ValueError(f"Header 'Time Relative (sec)' not found in {file_path}")
    df = pd.read_csv(file_path, sep="\t", decimal=",", skiprows=header_idx, engine="python")
    df.columns = [c.strip() for c in df.columns]
    return df


def clean_european_decimals(val):
    if isinstance(val, str):
        val = val.replace(",", ".")
        try:
            return float(val)
        except ValueError:
            return val
    return val


def load_parameters_file(file_path):
    df = pd.read_csv(file_path, sep=";", encoding="ISO-8859-1")
    try:
        df = df.map(clean_european_decimals)          # pandas >= 2.1
    except AttributeError:
        df = df.applymap(clean_european_decimals)      # older pandas
    df.columns = [c.strip() for c in df.columns]
    return df


# =============================================================================
# 3. PULSE-PERIOD DETECTION + ROLLING AVERAGE
# =============================================================================
def detect_pulse_period(df, time_col, pulse_col=PULSE_REFERENCE_SPECIES, min_gap_sec=MIN_PEAK_GAP_SEC, height_frac=0.5):
    """Median time between pulse peaks in `pulse_col` - used to size the
    rolling-average window to a whole number of cycles. CH4 is the natural
    choice since it's the actively valve-switched species."""
    t = df[time_col].to_numpy()
    sig = df[pulse_col].to_numpy()
    dt = np.median(np.diff(t))
    min_dist_pts = max(1, int(round(min_gap_sec / dt)))
    peaks, _ = find_peaks(sig, height=np.nanmax(sig) * height_frac, distance=min_dist_pts)
    if len(peaks) < 2:
        return None
    return float(np.median(np.diff(t[peaks])))


def add_rolling_average(df, time_col, species_map, n_cycles=N_CYCLES, period_sec=None):
    """Add '<col>_roll' columns using a centered rolling mean whose window
    spans n_cycles full pulse periods. Returns (df, window_points, window_sec,
    period_sec_used)."""
    if period_sec is None:
        period_sec = detect_pulse_period(df, time_col)
    if period_sec is None or period_sec <= 0:
        raise ValueError("Could not detect a pulse period - check the CH4 signal.")

    dt = df[time_col].diff().median()
    window_sec = n_cycles * period_sec
    win_points = max(1, int(round(window_sec / dt)))
    # require at least half the window present so edge points aren't
    # averaged over too few samples (they'll come out as NaN instead -
    # trimmed later rather than silently under-smoothed)
    min_periods = max(1, win_points // 2)

    for col in species_map:
        df[col + "_roll"] = df[col].rolling(win_points, center=True, min_periods=min_periods).mean()

    return df, win_points, window_sec, period_sec


# =============================================================================
# 4. MERGE REACTION + PARAMETERS  (reaction file itself is never modified)
# =============================================================================
def merge_reaction_with_parameters(reaction_df, param_df):
    reaction_df = reaction_df.rename(columns={"Time Relative (sec)": "Time_s"})
    param_df = param_df.rename(columns={"Dauer (s)": "Time_s"})
    reaction_df["Time_s"] = reaction_df["Time_s"].astype(float)
    param_df["Time_s"] = param_df["Time_s"].astype(float)

    # interpolate the (coarser) parameter log onto its own dense time axis
    param_interp = param_df.set_index("Time_s").sort_index().interpolate(method="linear").reset_index()

    # merge_asof only ADDS matched parameter columns to reaction_df; every
    # existing reaction_df column/value is left untouched
    merged = pd.merge_asof(
        reaction_df.sort_values("Time_s"),
        param_interp.sort_values("Time_s"),
        on="Time_s",
        direction="nearest",
    )
    return merged


# =============================================================================
# 5. KPI CALCULATION - FROM ROLLING-AVERAGED SIGNALS
# =============================================================================
def compute_kpis(df, bypass_avg):
    """Same formulas as the original master pipeline, but every signal
    reference points at the '_roll' (rolling-averaged) column instead of
    the raw one. bypass_avg is itself computed from the rolling-averaged
    bypass data (robust to the transient spikes at the start of a run)."""
    s = {name: df[f"Sig:_MCD_{name}_roll"] for name in ["CH4", "CO2", "H2", "CO", "H2O"]}

    df["X_CH4"] = (1 - (s["CH4"] / bypass_avg["CH4"])) * 100
    df["X_CO2"] = (1 - (s["CO2"] / bypass_avg["CO2"])) * 100
    df["Y_H2"] = (s["H2"] / (2 * bypass_avg["CH4"])) * 100

    total_carbon_in = bypass_avg["CH4"] + bypass_avg["CO2"]
    df["Y_CO"] = (s["CO"] / total_carbon_in) * 100
    df["Y_H2O"] = (s["H2O"] / (2 * bypass_avg["CH4"])) * 100

    df["Reacted_CH4"] = bypass_avg["CH4"] - s["CH4"]
    df["Reacted_CO2"] = bypass_avg["CO2"] - s["CO2"]
    df["S_H2"] = (s["H2"] / (2 * df["Reacted_CH4"])) * 100
    df["S_CO"] = (s["CO"] / (df["Reacted_CH4"] + df["Reacted_CO2"])) * 100
    df["S_H2"] = df["S_H2"].replace([np.inf, -np.inf], np.nan).fillna(0)
    df["S_CO"] = df["S_CO"].replace([np.inf, -np.inf], np.nan).fillna(0)

    df["H2_CO_Ratio"] = (s["H2"] / s["CO"]).replace([np.inf, -np.inf], np.nan).fillna(0)
    df["Carbon_Balance_%"] = ((s["CH4"] + s["CO2"] + s["CO"]) / total_carbon_in) * 100

    kpi_pct_cols = ["X_CH4", "X_CO2", "Y_H2", "Y_CO", "Y_H2O", "S_H2", "S_CO"]
    df[kpi_pct_cols] = df[kpi_pct_cols].clip(lower=0, upper=100)
    df["Carbon_Balance_%"] = df["Carbon_Balance_%"].clip(lower=0)
    return df


# =============================================================================
# 6. QC PLOTS - raw vs rolling average
# =============================================================================
def plot_bypass_qc(df, cat_id, win_points, window_sec, out_path):
    t = df["Time Relative (sec)"]
    fig, axes = plt.subplots(len(SPECIES), 1, figsize=(12, 12), sharex=True)
    for ax, (col, label) in zip(axes, SPECIES.items()):
        ax.plot(t, df[col], color=COLORS[label], lw=0.4, alpha=0.35, label="raw")
        ax.plot(t, df[col + "_roll"], color=COLORS[label], lw=1.4, label="rolling avg")
        ax.set_ylabel(f"{label}\n(a.u.)", fontsize=9)
        ax.grid(alpha=0.3)
    axes[0].legend(loc="upper right", fontsize=8, ncol=2)
    axes[-1].set_xlabel("Time relative (s)")
    fig.suptitle(f"{cat_id} - Bypass: raw vs rolling avg ({win_points} pts / {window_sec:.1f}s)", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_reaction_qc(df, cat_id, win_points, window_sec, out_path):
    t = df["Time_s"]
    t_min, t_max = t.min(), t.max()
    total = t_max - t_min
    w = total * ZOOM_WIDTH_FRAC
    windows = {
        "Start": (t_min, t_min + w),
        "Middle": (t_min + total / 2 - w / 2, t_min + total / 2 + w / 2),
        "End": (t_max - w, t_max),
    }
    fig, axes = plt.subplots(len(SPECIES), 3, figsize=(16, 14), sharey="row")
    for row, (col, label) in enumerate(SPECIES.items()):
        for c_idx, (wname, (lo, hi)) in enumerate(windows.items()):
            ax = axes[row, c_idx]
            mask = (t >= lo) & (t <= hi)
            ax.plot(t[mask], df.loc[mask, col], color=COLORS[label], lw=0.5, alpha=0.35, label="raw")
            ax.plot(t[mask], df.loc[mask, col + "_roll"], color=COLORS[label], lw=1.5, label="rolling avg")
            ax.grid(alpha=0.3)
            if row == 0:
                ax.set_title(f"{wname}\n[{lo:.0f}-{hi:.0f} s]", fontsize=10)
            if c_idx == 0:
                ax.set_ylabel(f"{label}\n(a.u.)", fontsize=9)
            if row == len(SPECIES) - 1:
                ax.set_xlabel("Time (s)")
            if row == 0 and c_idx == 0:
                ax.legend(loc="upper right", fontsize=7)
    fig.suptitle(f"{cat_id} - Reaction: raw vs rolling avg ({win_points} pts / {window_sec:.1f}s)", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


# =============================================================================
# 7. MAIN PER-CATALYST LOOP
# =============================================================================
def process_catalyst(cat):
    cat_id = cat["id"]
    print(f"\u23f3 Processing {cat_id} ...")

    bypass_df = load_mass_spec_file(cat["bypass"])
    reaction_df = load_mass_spec_file(cat["reaction"])
    param_df = load_parameters_file(cat["parameters"])

    # --- STEP 2: merge reaction + parameters (reaction values untouched) ---
    # NOTE: parameter columns (TIR-1, TIR-2, PIR-1, MFC-*, ...) are only
    # interpolated onto the reaction time axis for alignment purposes -
    # they are NEVER passed through add_rolling_average(). They stay
    # exactly as measured.
    merged_df = merge_reaction_with_parameters(reaction_df, param_df)
    param_cols = [c for c in param_df.columns if c != "Dauer (s)"]
    print(f"   parameters merged as-is (NOT smoothed): {param_cols}")

    # --- STEP 3: rolling average (species pulse period auto-detected) ---
    # The gas-switching valve timing is a fixed instrument setting - it does
    # NOT change between bypass and reaction. Bypass is the cleaner signal
    # (no catalytic reaction / no reaction-driven pressure spikes), so its
    # auto-detected period is the trustworthy reference. The reaction file's
    # OWN period is still detected independently purely as a diagnostic - if
    # it disagrees a lot with bypass, that's flagged, but the WINDOW itself
    # is always sized from the bypass period for both files, so bypass and
    # reaction are smoothed with a perfectly consistent, whole-cycle window.
    bypass_df, b_win_pts, b_win_sec, b_period = add_rolling_average(
        bypass_df, "Time Relative (sec)", SPECIES
    )

    try:
        r_period_check = detect_pulse_period(merged_df, "Time_s")
    except Exception:
        r_period_check = None

    if b_period is None:
        # safety fallback: bypass period detection failed (e.g. too few
        # pulses in a very short bypass run) - fall back to reaction's own
        # estimate rather than crashing
        if r_period_check is None:
            raise ValueError(f"{cat_id}: could not detect a pulse period from EITHER bypass or reaction.")
        print(f"   \u26a0\ufe0f  bypass period detection failed - falling back to reaction's own estimate ({r_period_check:.2f}s)")
        b_period = r_period_check
        bypass_df, b_win_pts, b_win_sec, b_period = add_rolling_average(
            bypass_df, "Time Relative (sec)", SPECIES, period_sec=b_period
        )
    elif r_period_check is not None and abs(r_period_check - b_period) / b_period > 0.10:
        print(f"   \u26a0\ufe0f  reaction-only period estimate ({r_period_check:.2f}s) disagrees with "
              f"bypass ({b_period:.2f}s) by >10% - likely noise/spikes splitting false peaks in the "
              f"reaction CH4 signal. Using the bypass period for the reaction window (valve timing is "
              f"shared hardware and doesn't change with reaction conditions).")

    merged_df, r_win_pts, r_win_sec, r_period = add_rolling_average(
        merged_df, "Time_s", SPECIES, period_sec=b_period  # <- reuse bypass period, not reaction's own
    )
    print(f"   pulse period used for BOTH windows (from bypass): {b_period:.2f}s "
          f"(reaction's own noisy estimate was {r_period_check if r_period_check is None else round(r_period_check,2)}s)")
    print(f"   -> window: bypass={b_win_sec:.1f}s ({b_win_pts}pt)  reaction={r_win_sec:.1f}s ({r_win_pts}pt)")

    merged_df["Catalyst_ID"] = cat_id
    merged_df["Time_min"] = merged_df["Time_s"] / 60.0

    # --- STEP 4: QC plots - generated (and saved) BEFORE any KPI is
    # calculated, so smoothing quality can be checked first ---
    bypass_qc_path = os.path.join(OUTPUT_FOLDER, f"{cat_id}_averaging_qc_bypass.png")
    reaction_qc_path = os.path.join(OUTPUT_FOLDER, f"{cat_id}_averaging_qc_reaction.png")
    plot_bypass_qc(bypass_df, cat_id, b_win_pts, b_win_sec, bypass_qc_path)
    plot_reaction_qc(merged_df, cat_id, r_win_pts, r_win_sec, reaction_qc_path)
    print(f"   QC plots saved -> {os.path.basename(bypass_qc_path)}, {os.path.basename(reaction_qc_path)}")

    # --- STEP 5: KPIs from rolling-averaged data ---
    # bypass baseline = mean of the ROLLED bypass signal (drop edge NaNs),
    # more robust to transient start-up spikes than a plain raw mean
    bypass_avg = {name: bypass_df[f"Sig:_MCD_{name}_roll"].mean(skipna=True) for name in ["CH4", "CO2", "H2", "CO", "H2O"]}
    merged_df = compute_kpis(merged_df, bypass_avg)

    # --- per-catalyst Excel export ---
    temp_col = next(c for c in merged_df.columns if "TIR-1" in c or "TIR_01" in c)
    kpi_export_cols = ["Time_s", "Time_min", temp_col, "X_CH4", "X_CO2", "Y_H2", "Y_CO", "Y_H2O",
                        "S_H2", "S_CO", "H2_CO_Ratio", "Carbon_Balance_%"]
    out_xlsx = os.path.join(OUTPUT_FOLDER, f"{cat_id}_integrated_results.xlsx")
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        merged_df.to_excel(writer, sheet_name="Sheet1_Combined_Data", index=False)
        bypass_df.to_excel(writer, sheet_name="Sheet2_Bypass_Raw+Rolled", index=False)
        merged_df[kpi_export_cols].to_excel(writer, sheet_name="Sheet3_Calculated_KPIs", index=False)

    qc_paths = {"bypass": bypass_qc_path, "reaction": reaction_qc_path}
    return merged_df, qc_paths


# =============================================================================
# 8. PLOTLY DASHBOARDS
#    Section A - TIME-DEPENDENT   (x = Time_min, dual y-axis, Temp on right)
#    Section B - TEMPERATURE-DEPENDENT (x = Temperature, single y-axis)
#    Both built from the same KPI set: Conversions, Yields, H2/CO Ratio,
#    Selectivities, Carbon Balance.
# =============================================================================
def _get_color_map(unique_catalysts):
    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#e377c2"]
    return {cat: palette[i % len(palette)] for i, cat in enumerate(unique_catalysts)}


def build_time_dependent_dashboard(master_df, temp_col, unique_catalysts, color_map):
    """Every KPI plotted vs Time_min, with Temperature on a secondary
    (right-hand) y-axis - the same pattern as the original 'stability'
    plot, generalized to conversions/yields/ratio/selectivity/carbon
    balance instead of just CO yield."""
    figs = []

    def add_temp_trace(fig, d, cat, row=1, col=1):
        fig.add_trace(go.Scatter(
            x=d["Time_min"], y=d[temp_col], mode="lines", name=f"{cat} Bed Temp", legendgroup=cat,
            showlegend=False, line=dict(color="#b0b0b0", dash="dot", width=1.2), hoverinfo="skip",
        ), row=row, col=col, secondary_y=True)

    # --- Conversions (CH4, CO2) ---
    fig_conv = make_subplots(rows=1, cols=2, subplot_titles=("Methane (CH4) Conversion", "Carbon Dioxide (CO2) Conversion"),
                              specs=[[{"secondary_y": True}, {"secondary_y": True}]])
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by="Time_min")
        fig_conv.add_trace(go.Scatter(x=d["Time_min"], y=d["X_CH4"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="CH4 Conversion: %{y:.1f}%<extra></extra>"), row=1, col=1, secondary_y=False)
        add_temp_trace(fig_conv, d, cat, 1, 1)
        fig_conv.add_trace(go.Scatter(x=d["Time_min"], y=d["X_CO2"], mode="lines", name=cat, legendgroup=cat, showlegend=False, line=dict(color=color_map[cat], width=2.5), hovertemplate="CO2 Conversion: %{y:.1f}%<extra></extra>"), row=1, col=2, secondary_y=False)
        add_temp_trace(fig_conv, d, cat, 1, 2)
    fig_conv.update_layout(title="<b>Conversions vs Time (rolling-averaged data)</b>", xaxis_title="Time (min)", xaxis2_title="Time (min)", template="plotly_white", hovermode="x unified")
    fig_conv.update_yaxes(title_text="Conversion (%)", range=[-5, 105], secondary_y=False, row=1, col=1)
    fig_conv.update_yaxes(title_text=f"Temp ({temp_col})", secondary_y=True, row=1, col=1)
    fig_conv.update_yaxes(title_text="Conversion (%)", range=[-5, 105], secondary_y=False, row=1, col=2)
    fig_conv.update_yaxes(title_text=f"Temp ({temp_col})", secondary_y=True, row=1, col=2)
    figs.append(fig_conv)

    # --- Yields (H2, CO, H2O) ---
    fig_yield = make_subplots(rows=1, cols=3, subplot_titles=("H2 Yield", "CO Yield", "H2O Yield Byproduct"),
                               specs=[[{"secondary_y": True}, {"secondary_y": True}, {"secondary_y": True}]])
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by="Time_min")
        fig_yield.add_trace(go.Scatter(x=d["Time_min"], y=d["Y_H2"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="H2 Yield: %{y:.1f}%<extra></extra>"), row=1, col=1, secondary_y=False)
        add_temp_trace(fig_yield, d, cat, 1, 1)
        fig_yield.add_trace(go.Scatter(x=d["Time_min"], y=d["Y_CO"], mode="lines", name=cat, legendgroup=cat, showlegend=False, line=dict(color=color_map[cat], width=2.5), hovertemplate="CO Yield: %{y:.1f}%<extra></extra>"), row=1, col=2, secondary_y=False)
        add_temp_trace(fig_yield, d, cat, 1, 2)
        fig_yield.add_trace(go.Scatter(x=d["Time_min"], y=d["Y_H2O"], mode="lines", name=cat, legendgroup=cat, showlegend=False, line=dict(color=color_map[cat], width=2.5, dash="dash"), hovertemplate="H2O Yield: %{y:.1f}%<extra></extra>"), row=1, col=3, secondary_y=False)
        add_temp_trace(fig_yield, d, cat, 1, 3)
    fig_yield.update_layout(title="<b>Product Yields vs Time (rolling-averaged data)</b>", xaxis_title="Time (min)", xaxis2_title="Time (min)", xaxis3_title="Time (min)", template="plotly_white", hovermode="x unified")
    for c in (1, 2, 3):
        fig_yield.update_yaxes(title_text="Yield (%)", range=[-5, 105], secondary_y=False, row=1, col=c)
        fig_yield.update_yaxes(title_text=f"Temp ({temp_col})", secondary_y=True, row=1, col=c)
    figs.append(fig_yield)

    # --- H2/CO Ratio ---
    fig_ratio = make_subplots(specs=[[{"secondary_y": True}]])
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by="Time_min")
        fig_ratio.add_trace(go.Scatter(x=d["Time_min"], y=d["H2_CO_Ratio"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="H2/CO Ratio: %{y:.2f}<extra></extra>"), secondary_y=False)
        add_temp_trace(fig_ratio, d, cat)
    fig_ratio.update_layout(title="<b>Syngas Quality (H2/CO Ratio) vs Time (rolling-averaged data)</b>", xaxis_title="Time (min)", template="plotly_white", hovermode="x unified")
    fig_ratio.update_yaxes(title_text="Molar Ratio (H2/CO)", secondary_y=False)
    fig_ratio.update_yaxes(title_text=f"Temp ({temp_col})", secondary_y=True)
    figs.append(fig_ratio)

    # --- Selectivities (H2, CO) ---
    fig_sel = make_subplots(rows=1, cols=2, subplot_titles=("H2 Selectivity", "CO Selectivity"),
                             specs=[[{"secondary_y": True}, {"secondary_y": True}]])
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by="Time_min")
        fig_sel.add_trace(go.Scatter(x=d["Time_min"], y=d["S_H2"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="H2 Selectivity: %{y:.1f}%<extra></extra>"), row=1, col=1, secondary_y=False)
        add_temp_trace(fig_sel, d, cat, 1, 1)
        fig_sel.add_trace(go.Scatter(x=d["Time_min"], y=d["S_CO"], mode="lines", name=cat, legendgroup=cat, showlegend=False, line=dict(color=color_map[cat], width=2.5), hovertemplate="CO Selectivity: %{y:.1f}%<extra></extra>"), row=1, col=2, secondary_y=False)
        add_temp_trace(fig_sel, d, cat, 1, 2)
    fig_sel.update_layout(title="<b>Product Selectivities vs Time (rolling-averaged data)</b>", xaxis_title="Time (min)", xaxis2_title="Time (min)", template="plotly_white", hovermode="x unified")
    for c in (1, 2):
        fig_sel.update_yaxes(title_text="Selectivity (%)", range=[-5, 105], secondary_y=False, row=1, col=c)
        fig_sel.update_yaxes(title_text=f"Temp ({temp_col})", secondary_y=True, row=1, col=c)
    figs.append(fig_sel)

    # --- Carbon Balance ---
    fig_carbon = make_subplots(specs=[[{"secondary_y": True}]])
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by="Time_min")
        fig_carbon.add_trace(go.Scatter(x=d["Time_min"], y=d["Carbon_Balance_%"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="Carbon Balance: %{y:.1f}%<extra></extra>"), secondary_y=False)
        add_temp_trace(fig_carbon, d, cat)
    fig_carbon.update_layout(title="<b>Total Gas-Phase Carbon Balance vs Time (rolling-averaged data)</b>", xaxis_title="Time (min)", template="plotly_white", hovermode="x unified")
    fig_carbon.update_yaxes(title_text="Carbon Balance (%)", secondary_y=False)
    fig_carbon.update_yaxes(title_text=f"Temp ({temp_col})", secondary_y=True)
    figs.append(fig_carbon)

    return figs


def build_temperature_dependent_dashboard(master_df, temp_col, unique_catalysts, color_map):
    """Every KPI plotted vs reactor Temperature, single y-axis - the
    original 5-row comparison dashboard from the master pipeline."""
    figs = []

    fig_conv = make_subplots(rows=1, cols=2, subplot_titles=("Methane (CH4) Conversion", "Carbon Dioxide (CO2) Conversion"), shared_yaxes=True)
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by=temp_col)
        fig_conv.add_trace(go.Scatter(x=d[temp_col], y=d["X_CH4"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="CH4 Conversion: %{y:.1f}%<extra></extra>"), row=1, col=1)
        fig_conv.add_trace(go.Scatter(x=d[temp_col], y=d["X_CO2"], mode="lines", name=cat, legendgroup=cat, showlegend=False, line=dict(color=color_map[cat], width=2.5), hovertemplate="CO2 Conversion: %{y:.1f}%<extra></extra>"), row=1, col=2)
    fig_conv.update_layout(title="<b>Reactor Conversion Profiles vs Temperature (rolling-averaged data)</b>", xaxis_title=f"Temperature ({temp_col})", xaxis2_title=f"Temperature ({temp_col})", yaxis_title="Conversion (%)", yaxis_range=[-5, 105], template="plotly_white", hovermode="x unified")
    figs.append(fig_conv)

    fig_yield = make_subplots(rows=1, cols=3, subplot_titles=("H2 Yield", "CO Yield", "H2O Yield Byproduct"), shared_yaxes=True)
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by=temp_col)
        fig_yield.add_trace(go.Scatter(x=d[temp_col], y=d["Y_H2"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="H2 Yield: %{y:.1f}%<extra></extra>"), row=1, col=1)
        fig_yield.add_trace(go.Scatter(x=d[temp_col], y=d["Y_CO"], mode="lines", name=cat, legendgroup=cat, showlegend=False, line=dict(color=color_map[cat], width=2.5), hovertemplate="CO Yield: %{y:.1f}%<extra></extra>"), row=1, col=2)
        fig_yield.add_trace(go.Scatter(x=d[temp_col], y=d["Y_H2O"], mode="lines", name=cat, legendgroup=cat, showlegend=False, line=dict(color=color_map[cat], width=2.5, dash="dash"), hovertemplate="H2O Yield: %{y:.1f}%<extra></extra>"), row=1, col=3)
    fig_yield.update_layout(title="<b>Product Synthesis Yield Matrix vs Temperature (rolling-averaged data)</b>", xaxis_title=temp_col, xaxis2_title=temp_col, xaxis3_title=temp_col, yaxis_title="Yield (%)", yaxis_range=[-5, 105], template="plotly_white", hovermode="x unified")
    figs.append(fig_yield)

    fig_ratio = go.Figure()
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by=temp_col)
        fig_ratio.add_trace(go.Scatter(x=d[temp_col], y=d["H2_CO_Ratio"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="H2/CO Ratio: %{y:.2f}<extra></extra>"))
    fig_ratio.update_layout(title="<b>Syngas Quality (H2/CO Ratio) vs Temperature (rolling-averaged data)</b>", xaxis_title=f"Reactor Temperature ({temp_col})", yaxis_title="Molar Ratio (H2/CO)", template="plotly_white", hovermode="x unified")
    figs.append(fig_ratio)

    fig_sel = make_subplots(rows=1, cols=2, subplot_titles=("H2 Selectivity", "CO Selectivity"), shared_yaxes=True)
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by=temp_col)
        fig_sel.add_trace(go.Scatter(x=d[temp_col], y=d["S_H2"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="H2 Selectivity: %{y:.1f}%<extra></extra>"), row=1, col=1)
        fig_sel.add_trace(go.Scatter(x=d[temp_col], y=d["S_CO"], mode="lines", name=cat, legendgroup=cat, showlegend=False, line=dict(color=color_map[cat], width=2.5), hovertemplate="CO Selectivity: %{y:.1f}%<extra></extra>"), row=1, col=2)
    fig_sel.update_layout(title="<b>Product Selectivities vs Temperature (rolling-averaged data)</b>", xaxis_title=temp_col, xaxis2_title=temp_col, yaxis_title="Selectivity (%)", yaxis_range=[-5, 105], template="plotly_white", hovermode="x unified")
    figs.append(fig_sel)

    fig_carbon = go.Figure()
    for cat in unique_catalysts:
        d = master_df[master_df["Catalyst_ID"] == cat].sort_values(by=temp_col)
        fig_carbon.add_trace(go.Scatter(x=d[temp_col], y=d["Carbon_Balance_%"], mode="lines", name=cat, legendgroup=cat, line=dict(color=color_map[cat], width=2.5), hovertemplate="Carbon Balance: %{y:.1f}%<extra></extra>"))
    fig_carbon.update_layout(title="<b>Total Gas-Phase Carbon Balance vs Temperature (rolling-averaged data)</b>", xaxis_title=f"Reactor Temperature ({temp_col})", yaxis_title="Carbon Balance (%)", template="plotly_white", hovermode="x unified")
    figs.append(fig_carbon)

    return figs


def _img_to_base64_html(path, alt):
    import base64
    with open(path, "rb") as fh:
        b64 = base64.b64encode(fh.read()).decode("ascii")
    return f'<img src="data:image/png;base64,{b64}" alt="{alt}" style="max-width:100%; margin-bottom:20px;"/>'


def compile_dashboard(master_df, qc_paths_by_catalyst, out_html):
    unique_catalysts = sorted(master_df["Catalyst_ID"].unique())
    temp_col = next(c for c in master_df.columns if "TIR-1" in c or "TIR_01" in c)
    color_map = _get_color_map(unique_catalysts)

    time_figs = build_time_dependent_dashboard(master_df, temp_col, unique_catalysts, color_map)
    temp_figs = build_temperature_dependent_dashboard(master_df, temp_col, unique_catalysts, color_map)

    with open(out_html, "w", encoding="utf-8") as f:
        f.write("<html><head><title>DRM Interactive Dashboard (rolling-averaged)</title></head><body "
                "style='font-family:Arial, sans-serif; max-width:1400px; margin:0 auto;'>\n")

        # --- QC section: rolling-average smoothing check, per catalyst ---
        f.write("<h1>Rolling-Average QC (raw vs smoothed - check before trusting KPIs below)</h1>\n")
        for cat in unique_catalysts:
            f.write(f"<h3>{cat}</h3>\n")
            f.write(_img_to_base64_html(qc_paths_by_catalyst[cat]["bypass"], f"{cat} bypass QC"))
            f.write(_img_to_base64_html(qc_paths_by_catalyst[cat]["reaction"], f"{cat} reaction QC"))

        # --- Section A: time-dependent ---
        f.write("<h1>Time-Dependent Plots (vs Time, Temperature on right axis)</h1>\n")
        for i, fig in enumerate(time_figs):
            f.write(fig.to_html(full_html=False, include_plotlyjs=("cdn" if i == 0 else False)))

        # --- Section B: temperature-dependent ---
        f.write("<h1>Temperature-Dependent Plots (vs Temperature)</h1>\n")
        for fig in temp_figs:
            f.write(fig.to_html(full_html=False, include_plotlyjs=False))

        f.write("</body></html>")

    return time_figs, temp_figs


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    catalysts = discover_catalysts(INPUT_FOLDER)
    print(f"\U0001F4CA Discovered {len(catalysts)} catalyst dataset(s): {[c['id'] for c in catalysts]}")

    all_dfs = []
    qc_paths_by_catalyst = {}
    for cat in catalysts:
        df, qc_paths = process_catalyst(cat)
        all_dfs.append(df)
        qc_paths_by_catalyst[cat["id"]] = qc_paths

    master_df = pd.concat(all_dfs, ignore_index=True)
    temp_col = next(c for c in master_df.columns if "TIR-1" in c or "TIR_01" in c)
    looker_cols = ["Catalyst_ID", "Time_s", "Time_min", temp_col, "X_CH4", "X_CO2", "Y_H2",
                   "Y_CO", "Y_H2O", "S_H2", "S_CO", "H2_CO_Ratio", "Carbon_Balance_%"]
    master_csv = os.path.join(OUTPUT_FOLDER, "Master_DRM_LookerStudio_Data.csv")
    master_df[looker_cols].to_csv(master_csv, index=False)
    print(f"\U0001F5C2\uFE0F Master CSV -> {master_csv}")

    dashboard_html = os.path.join(OUTPUT_FOLDER, "DRM_Interactive_Dashboard.html")
    compile_dashboard(master_df, qc_paths_by_catalyst, dashboard_html)
    print(f"\U0001F3C1 Dashboard -> {dashboard_html}")
